In [3]:
import cv2, glob, os, sys
import tqdm
import PIL
import torch
import numpy as np
from PIL import Image
from pathlib import Path
import torchvision.transforms as transforms

# Configuración de GPU
torch.cuda.set_device(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch._C._jit_set_profiling_mode(False)

current_path = Path(os.getcwd())
# Simulamos el .parent.parent.parent del script original
root_path = str(current_path.parent.parent.parent)

if root_path not in sys.path:
    sys.path.insert(0, root_path)
# ---------------------------

from config.config import PATH_DATASET


ModuleNotFoundError: No module named 'config'

In [ ]:
def transform_image(image):
    to_tensor_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )])
    return to_tensor_norm(image).unsqueeze(0).to(device)

def predict_fragmentation(model, image_path, shape):
    img = Image.open(image_path).convert("RGB")
    orig_w, orig_h = img.size
    image = transforms.Resize(shape)(img)
    input_tensor = transform_image(image)
    
    with torch.no_grad():
        outputs = model(input_tensor)

    outputs = torch.argmax(outputs, 1)
    mask = outputs.squeeze(0).cpu().numpy()
    mask[mask == 1] = 255
    return np.array(transforms.Resize((orig_h, orig_w))(Image.fromarray(mask.astype('uint8'))))